# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barak-hub/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Section 1 — Two Paper Findings and Methodology Questions

**Finding 1: Random Forest Feature Importance (Page 27)**

The FlyRank research paper reports that average position contributes 43% and impressions contribute 32% of the Random Forest feature importance when predicting the FlyRank health score.

My methodology concern is that the health score is already calculated using impressions, average position, CTR, and scroll depth. Therefore, using these same variables to predict the health score creates a circular prediction problem and potential target leakage.

**Methodology question:** Can this model demonstrate meaningful predictive performance when evaluated against an independent outcome, rather than a score constructed from its input features?

**Finding 2: Logistic Regression Growth Prediction (Page 29)**

The research reports 71% holdout accuracy for a logistic regression model classifying growing and declining content.

Although this provides an initial performance estimate, the paper does not demonstrate that the evaluation used a grouped or time-aware split. A random split may overestimate generalization if related pages or overlapping reporting periods appear in both training and testing datasets.

**Methodology question:** Would the model maintain comparable performance when evaluated using a time-aware split or when entire brands are held out from training?

**Conclusion**

Both findings are useful for exploratory analysis, but their interpretation depends on how the target labels are constructed and how the models are validated. Independent evaluation, leakage checks, and appropriate train/test separation are necessary before making strong predictive claims.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:

import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

token = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=token
)

df = pd.read_parquet(file_path)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
print("Unique webpages:", df[
    ["client_hash_id", "content_hash_id"]
].drop_duplicates().shape[0])


Dataset loaded successfully!
Rows: 2414248
Unique clients: 52
Unique webpages: 133852


In [10]:

# Reconstruct the Week 5 webpage-level dataset

page_df = (
    df.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions_prev30=("impressions_prev30", "sum"),
        impressions_last30=("impressions_last30", "sum"),
        clicks_prev30=("clicks_prev30", "sum"),
        query_count=("query_hash_id", "nunique")
    )
)

# Create the observed decline label
page_df["declining_impressions"] = (
    page_df["impressions_last30"]
    < page_df["impressions_prev30"]
).astype(int)

print("Total webpages:", len(page_df))

print("\nTarget distribution:")
print(page_df["declining_impressions"].value_counts())

print("\nMissing values:")
print(page_df.isna().sum())


Total webpages: 133852

Target distribution:
declining_impressions
1    80534
0    53318
Name: count, dtype: int64

Missing values:
client_hash_id           0
content_hash_id          0
impressions_prev30       0
impressions_last30       0
clicks_prev30            0
query_count              0
declining_impressions    0
dtype: int64


In [11]:

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

# Use the same two features as Week 5
features = ["impressions_prev30", "clicks_prev30"]

X = page_df[features]
y = page_df["declining_impressions"]

# Random webpage-level split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Train Decision Tree
model_random = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model_random.fit(X_train, y_train)

# Evaluate
y_pred = model_random.predict(X_test)

print("Random Split Results")
print("--------------------")
print("Training webpages:", len(X_train))
print("Testing webpages:", len(X_test))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1-score:", round(f1_score(y_test, y_pred), 4))

# Check client overlap
train_clients = set(page_df.loc[X_train.index, "client_hash_id"])
test_clients = set(page_df.loc[X_test.index, "client_hash_id"])

print("Clients in both groups:", len(train_clients & test_clients))


Random Split Results
--------------------
Training webpages: 107081
Testing webpages: 26771
Precision: 0.6925
Recall: 0.954
F1-score: 0.8025
Clients in both groups: 48


In [12]:

print("Random split model parameters:")
print(model_random.get_params())


Random split model parameters:
{'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 5, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': 42, 'splitter': 'best'}


In [13]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

# Match the Week 5 model configuration
model_random_corrected = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=100,
    random_state=42
)

# Use the existing random split
model_random_corrected.fit(X_train, y_train)

# Generate predictions
y_pred_corrected = model_random_corrected.predict(X_test)

print("Corrected Random Split Results")
print("--------------------------------")
print("Precision:", round(
    precision_score(y_test, y_pred_corrected), 4
))
print("Recall:", round(
    recall_score(y_test, y_pred_corrected), 4
))
print("F1-score:", round(
    f1_score(y_test, y_pred_corrected), 4
))

print("\nModel configuration:")
print("max_depth:", model_random_corrected.max_depth)
print("min_samples_leaf:", model_random_corrected.min_samples_leaf)


Corrected Random Split Results
--------------------------------
Precision: 0.6914
Recall: 0.9569
F1-score: 0.8028

Model configuration:
max_depth: 4
min_samples_leaf: 100


### Section 2 — Model Validation Under Different Splits

**Objective**

I compared two validation strategies for my Decision Tree Classifier: a random webpage-level split and a client-grouped split. Both experiments used the same model configuration (`max_depth=4`, `min_samples_leaf=100`, `random_state=42`) and the same two input features: previous-period impressions and clicks.

**Results**

| Metric | Random Split | Client-Grouped Split |
|---|---:|---:|
| Precision | 0.6914 | 0.7262 |
| Recall | 0.9569 | 0.9378 |
| F1-score | 0.8028 | 0.8185 |
| Client overlap | 48 | 0 |

**Interpretation**

The random split produced an F1-score of 0.8028, while the client-grouped split achieved 0.8185, a difference of 0.0157.

The random split allowed 48 clients to appear in both training and testing. This creates a risk that client-specific patterns could influence evaluation results.

The client-grouped split prevented client overlap, providing a more appropriate assessment of generalization to previously unseen clients.

However, the two experiments used different test sets and training sizes. Therefore, the performance difference cannot be attributed exclusively to the splitting strategy or interpreted as direct evidence of leakage.

**Limitations**

Both experiments classify observed declines within a single historical reporting window. Neither establishes performance on future periods. Additional chronological validation would be required before claiming reliable forecasting performance.

**Conclusion**

The client-grouped evaluation provides stronger evidence of generalization across clients, even though the observed F1-score difference is relatively small. The experiment highlights why validation design matters alongside model performance metrics.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [15]:

# Week 6 — Section 3: Feature Leakage Audit

feature_columns = ["impressions_prev30", "clicks_prev30"]
target_column = "declining_impressions"

print("FEATURE LEAKAGE AUDIT")
print("---------------------")

print("Model features:", feature_columns)
print("Target:", target_column)

# Check whether last-period impressions are used as input
print(
    "Last30 impressions used as a feature:",
    "impressions_last30" in feature_columns
)

# Check whether the target itself is used as input
print(
    "Target used as a feature:",
    target_column in feature_columns
)

# Verify target construction
expected_target = (
    page_df["impressions_last30"]
    < page_df["impressions_prev30"]
).astype(int)

print(
    "Target construction matches:",
    expected_target.equals(page_df[target_column])
)

print(
    "Missing values in model features:",
    page_df[feature_columns].isna().sum().sum()
)


FEATURE LEAKAGE AUDIT
---------------------
Model features: ['impressions_prev30', 'clicks_prev30']
Target: declining_impressions
Last30 impressions used as a feature: False
Target used as a feature: False
Target construction matches: True
Missing values in model features: 0


In [16]:

# Week 6 — Section 3: Feature Timing Audit

audit = {
    "impressions_prev30": {
        "role": "Input feature",
        "period": "Previous 30 days",
        "available_before_outcome": True
    },
    "clicks_prev30": {
        "role": "Input feature",
        "period": "Previous 30 days",
        "available_before_outcome": True
    },
    "impressions_last30": {
        "role": "Target construction only",
        "period": "Last 30 days",
        "available_before_outcome": False
    }
}

timing_df = pd.DataFrame.from_dict(
    audit,
    orient="index"
)

print("FEATURE TIMING AUDIT")
print(timing_df.to_string())


FEATURE TIMING AUDIT
                                        role            period  available_before_outcome
impressions_prev30             Input feature  Previous 30 days                      True
clicks_prev30                  Input feature  Previous 30 days                      True
impressions_last30  Target construction only      Last 30 days                     False


### Section 3 — Data Leakage Audit

**Objective**

I audited my Decision Tree Classifier to identify potential data leakage, particularly whether information from the target period was improperly included in the model inputs.

**Audit Results**

| Check | Result |
|---|---|
| `impressions_last30` included in model features | No |
| `declining_impressions` included in model features | No |
| Target construction matches its definition | Yes |
| Missing values in input features | 0 |
| Client overlap in grouped validation | 0 |

**Feature Timing Assessment**

The model uses `impressions_prev30` and `clicks_prev30` as input features. Based on the column names, these represent historical measurements from the previous 30-day period.

The target is constructed by comparing `impressions_last30` against `impressions_prev30`. Therefore, the target cannot be determined until the last 30-day observation period is complete.

The model does not directly use last-period impressions as an input feature.

**Remaining Risks**

1. **Temporal leakage:** The audit assumes that previous-period measurements were available at prediction time. This has not yet been independently verified against source timestamps or data-generation documentation.
2. **Client-level contamination:** A random webpage split allows the same clients into training and testing. The client-grouped split avoids this overlap.
3. **Target definition:** The model predicts a decline relative to a historical baseline. Because that baseline is also an input feature, its importance may reflect the target's mathematical construction rather than an independent causal relationship.
4. **Limited temporal validation:** The available experiment uses a single 90-day reporting window and does not establish generalization across future time periods.

**Conclusion**

The audit found no direct inclusion of the target or last-period impressions among the model features. Client-grouped validation also prevents direct client overlap between training and testing.

However, these checks are insufficient to certify the model as leakage-free. Timestamp verification and chronological validation are necessary before making strong claims about future predictive performance.

### Section 4 — Claim Rewrite

**Original claim (too strong):**

"My machine learning model accurately predicts which webpages will lose search impressions in the future."

**Revised claim (evidence-based):**

"My Decision Tree Classifier identified patterns associated with observed declines in webpage search impressions within the available reporting window. Under client-grouped validation, the model achieved a measured F1-score of 0.8185 on held-out clients. These results provide directional insights that may support SEO decision-making, but they do not establish reliable future forecasting performance."

**Why I revised the claim:**

The original statement implies that the model can reliably predict future performance, which has not been demonstrated.

The revised statement distinguishes measured classification performance from forecasting capability, acknowledges the validation method, and presents the model as a decision-support tool rather than a guaranteed prediction system.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.